# VORTEXRAG — Interactive Demo

**Vector Orthogonal Resonance-Tuned EXtraction Retrieval-Augmented Generation**

> *The only RAG that kills semantic drift and context poisoning simultaneously.*

This notebook demonstrates the full VORTEXRAG 7-layer pipeline:
1. **TVE** — Tri-Vector Encoding (semantic + syntactic + causal)
2. **VRC** — Vortex Retrieval Cone (spiral topology ranking)
3. **SDC** — Semantic Drift Corrector (causal alignment gate)
4. **CPG** — Context Poison Guard (ESR-based window purification)
5. **RFG** — Rank Fusion Gate (Φ-score multiplicative ranking)
6. **CCB** — Causal Context Builder (causal depth ordering)
7. **FV**  — Faithfulness Verifier (ΔR hallucination detection)


## 1. Installation


In [ ]:
# Install VORTEXRAG and optional dependencies
!pip install numpy>=1.24.0 --quiet

# Optional: for full semantic embeddings (uncomment to use)
# !pip install sentence-transformers>=2.2.0 --quiet
# !pip install spacy>=3.5.0 --quiet
# !python -m spacy download en_core_web_sm

# Clone the repository if running in Colab
import os
if not os.path.exists('VORTEXRAG'):
    !git clone https://github.com/vignesh2027/VORTEXRAG.git --quiet
    os.chdir('VORTEXRAG')
elif os.path.basename(os.getcwd()) != 'VORTEXRAG':
    os.chdir('VORTEXRAG')

import sys
sys.path.insert(0, '.')
print('Setup complete!')


## 2. Quick Start


In [ ]:
from vortexrag import VortexRAG, VortexRAGConfig

# Sample corpus about the 2008 financial crisis
corpus = [
    """The 2008 financial crisis was triggered by the collapse of the housing bubble.
    Mortgage-backed securities and CDOs were overvalued, leading to massive losses
    when subprime mortgages defaulted at scale.""",

    """Lehman Brothers filed for bankruptcy on September 15, 2008, sending shockwaves
    through global markets. The bank had massive exposure to toxic mortgage-backed
    assets and could not meet its debt obligations.""",

    """The Federal Reserve responded by cutting interest rates to near zero and
    implementing quantitative easing programs to stabilize frozen credit markets.""",

    """Credit default swaps amplified systemic risk by creating synthetic exposure
    to subprime mortgages. When defaults rose, counterparties could not pay out,
    cascading losses through the financial system.""",
]

# Configure for financial domain
config = VortexRAGConfig(domain='financial', verbose=True)
config.vrc.top_k = 8
config.rfg.top_m = 4

# Initialize, index, and query
rag = VortexRAG(config=config)
rag.index(additional_texts=corpus)
result = rag.query('What caused the 2008 financial crisis?')

print(f'\nDelta_R (hallucination): {result.delta_r:.4f}')
print(f'ESR (signal ratio):      {result.esr:.4f}')
print(f'Latency:                 {result.latency_ms:.1f}ms')
print(f'Accepted:                {result.accepted}')


## 3. Layer-by-Layer Pipeline Trace

Inspect each layer's decisions step by step.


In [ ]:
from core.tve import TriVectorEncoder, TVEConfig, DOMAIN_WEIGHTS
from core.vrc import VortexRetrievalCone, VRCConfig
from core.sdc import SemanticDriftCorrector, SDCConfig
from core.cpg import ContextPoisonGuard, CPGConfig
from core.rfg import RankFusionGate, RFGConfig
from core.ccb import CausalContextBuilder, CCBConfig
from core.fv import FaithfulnessVerifier, FVConfig

query = 'What caused the 2008 financial crisis?'
domain = 'financial'

# Layer 2: TVE Encoding
a, b, g = DOMAIN_WEIGHTS[domain]
encoder = TriVectorEncoder(TVEConfig(alpha=a, beta=b, gamma=g, domain=domain))
q_vec = encoder.encode_query(query)
corpus_vecs = [encoder.encode_chunk(c) for c in corpus]
tve_scores = encoder.batch_tve_scores(q_vec, corpus_vecs)

print('=== LAYER 2: TRI-VECTOR ENCODING ===')
print(f'Query vector: {q_vec.dim}d ({q_vec.semantic_dim}d x 3 arms)')
print(f'Weights: alpha={a:.2f} (sem), beta={b:.2f} (syn), gamma={g:.2f} (cau)')
for i, (score, chunk) in enumerate(zip(tve_scores, corpus)):
    bar = 'X' * int(float(score) * 20)
    print(f'  [{i+1}] TVE={float(score):.3f} {bar} | {chunk[:55].strip()}...')


In [ ]:
# Layer 3: VRC Spiral Retrieval
vrc = VortexRetrievalCone(encoder, VRCConfig(n_spiral=2, lambda_decay=0.5, top_k=4))
spiral_pool = vrc.retrieve(q_vec, corpus_vecs, corpus)

print('=== LAYER 3: VORTEX RETRIEVAL CONE ===')
print(f'Spiral: n={vrc.config.n_spiral}, lambda={vrc.config.lambda_decay}')
for i, c in enumerate(spiral_pool):
    print(f'  [{i+1}] spiral_rank={c.spiral_rank:+.3f} | TVE={c.tve_score:.3f} | '
          f'theta={c.theta*57.3:.1f}deg | {c.chunk_text[:50].strip()}...')

stats = vrc.pool_statistics(spiral_pool)
if stats:
    print(f'\nPool stats: mean_TVE={stats["mean_tve"]:.3f}, suppressed={stats["n_negative_rank"]}/{stats["n_candidates"]}')


In [ ]:
# Layer 4a: SDC - Semantic Drift Corrector
sdc = SemanticDriftCorrector(SDCConfig(domain=domain))
sdc_results = sdc.filter(q_vec, spiral_pool)
accepted = sdc.accepted_only(sdc_results)

print('=== LAYER 4a: SEMANTIC DRIFT CORRECTOR ===')
print(f'tau={sdc.config.tau}, delta_SDC={sdc.config.delta_sdc}')
print(f'Accepted {len(accepted)}/{len(sdc_results)} chunks')
for r in sdc_results:
    mark = 'ACCEPT' if r.accepted else 'REJECT'
    print(f'  [{mark}] SDS={r.sds_score:.3f} ({r.drift_category}) | {r.candidate.chunk_text[:50].strip()}...')


In [ ]:
# Layer 4b: CPG - Context Poison Guard
cpg = ContextPoisonGuard(CPGConfig(theta_cpg=3.5, min_chunks=2))
working = accepted if len(accepted) >= 2 else sdc_results
cpg_eval = cpg.evaluate(q_vec, working)

print('=== LAYER 4b: CONTEXT POISON GUARD ===')
print(f'theta_CPG={cpg.config.theta_cpg}')
print(f'ESR = {cpg_eval.esr:.4f} ({"CLEAN" if cpg_eval.is_clean else "DIRTY"})')
print(f'Poison Index = {cpg_eval.poison_index:.4f}')
print(f'Chunks purged: {cpg_eval.purge_count}')
print(f'Window size: {len(cpg_eval.window)}')


In [ ]:
# Layers 5a+5b: RFG + CCB
rfg = RankFusionGate(RFGConfig(domain=domain, top_m=min(4, len(cpg_eval.window))))
rfg.config.apply_domain_preset()
ranked = rfg.rank(q_vec, cpg_eval)
final = rfg.select_top_m(ranked)

ccb = CausalContextBuilder(CCBConfig())
slots = ccb.build(query, final)
context_str = ccb.to_context_string(slots)

print('=== LAYER 5a: RANK FUSION GATE ===')
print(f'Phi weights: alpha={rfg.config.alpha}, beta={rfg.config.beta}, gamma={rfg.config.gamma}')
for i, c in enumerate(final):
    print(f'  [{i+1}] Phi_norm={c.phi_norm:.4f} | TVE={c.tve_score:.3f} | SDS={c.sds_score:.3f} | {c.chunk_text[:45].strip()}...')

print()
print('=== LAYER 5b: CAUSAL CONTEXT BUILDER ===')
for slot in slots:
    label = {0: 'ROOT', 1: 'EFFECT', 2: 'SUPPORT'}.get(slot.causal_depth, f'D{slot.causal_depth}')
    print(f'  [{label}] pos={slot.slot_position:.0f} | {slot.chunk.chunk_text[:50].strip()}...')


In [ ]:
# Layer 7: Faithfulness Verifier
fv = FaithfulnessVerifier(FVConfig(delta_fv=0.15, use_nli=False))

# Stub answer from context (replace with your real LLM in production)
stub_answer = context_str.split('\n\n')[0][:250] if context_str else 'No answer.'
fv_result = fv.verify(stub_answer, context_str)

print('=== LAYER 7: FAITHFULNESS VERIFIER ===')
print(f'Delta_R   = {fv_result.delta_r:.4f}  (threshold: delta_FV={fv.config.delta_fv})')
print(f'ROUGE-L   = {fv_result.rouge_l:.4f}')
print(f'Grounding = {fv_result.grounding:.4f}')
print(f'Verdict:    {"ACCEPTED" if fv_result.accepted else "REJECTED (regenerate)"}')


## 4. Domain Preset Comparison

Compare how different domain presets affect retrieval quality on the same corpus.


In [ ]:
from core.tve import DOMAIN_WEIGHTS

domains_to_compare = ['general', 'financial', 'legal', 'medical', 'scientific', 'code']
results_table = []

for dom in domains_to_compare:
    cfg = VortexRAGConfig(domain=dom, verbose=False)
    cfg.vrc.top_k = 4
    cfg.rfg.top_m = 3
    r_rag = VortexRAG(config=cfg)
    r_rag.index(additional_texts=corpus)
    res = r_rag.query('What caused the 2008 financial crisis?')
    a, b, g = DOMAIN_WEIGHTS[dom]
    results_table.append({'Domain': dom, 'alpha': a, 'beta': b, 'gamma': g,
                          'ESR': round(res.esr, 3), 'DeltaR': round(res.delta_r, 3),
                          'Purged': res.purge_count})

print(f'{"Domain":<14} {"alpha":>6} {"beta":>6} {"gamma":>6} {"ESR":>8} {"DeltaR":>8} {"Purged":>7}')
print('-' * 65)
for r in results_table:
    print(f'  {r["Domain"]:<12} {r["alpha"]:>6.2f} {r["beta"]:>6.2f} {r["gamma"]:>6.2f} '
          f'{r["ESR"]:>8.3f} {r["DeltaR"]:>8.3f} {r["Purged"]:>7}')


## 5. Ablation Study

Test what happens when pipeline layers are disabled to understand each layer's contribution.


In [ ]:
# Mixed corpus: clean + poisoned chunks
mixed_corpus = [
    'The 2008 crisis was caused by CDO derivatives and subprime mortgage defaults.',
    'Lehman Brothers collapse triggered a global financial panic in September 2008.',
    'Financial markets crashed due to toxic mortgage-backed securities exposure.',
    # Poison chunks below - semantically similar but causally unrelated
    'The stock market is a place where companies raise capital through equity issuance.',
    'Investment banking involves advisory services for mergers and acquisitions transactions.',
    'Economics is the study of how societies allocate scarce resources among competing uses.',
]

ablation_configs = [
    ('Full VORTEXRAG',  3.5,    0.72),
    ('No CPG (theta=0)', 0.001, 0.72),
    ('No SDC (delta=0)', 3.5,   0.01),
    ('No SDC+CPG',       0.001, 0.01),
]

print(f'{"Configuration":<22} {"ESR":>8} {"Delta_R":>8} {"Purged":>8} {"OK":>6}')
print('-' * 58)

for name, theta, delta_sdc in ablation_configs:
    cfg = VortexRAGConfig(domain='financial', verbose=False)
    cfg.vrc.top_k = min(len(mixed_corpus), 8)
    cfg.rfg.top_m = 4
    cfg.sdc.delta_sdc = delta_sdc
    cfg.cpg.theta_cpg = theta
    rag = VortexRAG(config=cfg)
    rag.index(additional_texts=mixed_corpus)
    r = rag.query('What caused the financial crisis?')
    print(f'  {name:<20} {r.esr:>8.3f} {r.delta_r:>8.3f} {r.purge_count:>8} {"YES" if r.accepted else "NO":>6}')


## 6. Benchmark Comparison (Mock)


In [ ]:
import re

def token_f1(pred, gold):
    def norm(s): return re.sub(r'[^\w\s]', '', s.lower()).split()
    p_tok, g_tok = norm(pred), norm(gold)
    if not p_tok or not g_tok: return 0.0
    common = set(p_tok) & set(g_tok)
    if not common: return 0.0
    pr = len(common) / len(p_tok)
    rc = len(common) / len(g_tok)
    return 2 * pr * rc / (pr + rc)

test_samples = [
    {'q': 'What caused the financial crisis?', 'gold': 'mortgage securities and subprime defaults',
     'chunks': ['Subprime mortgage defaults caused cascading MBS losses.', 'The housing bubble burst in 2007.',
                'General banking history is well documented across centuries.']},
    {'q': 'What is quantitative easing?', 'gold': 'central bank buying assets to inject money',
     'chunks': ['Quantitative easing means a central bank buying assets to inject money into the economy.',
                'The Fed cut rates to near zero during the crisis.', 'Stock markets respond to stimulus.']},
]

print(f'{"Method":<15} {"F1":>8} {"ESR":>8} {"Delta_R":>10}')
print('-' * 45)

for method_name, theta_cpg, delta_sdc in [('VORTEXRAG', 3.5, 0.72), ('Naive RAG', 0.001, 0.01)]:
    f1s, esrs, drs = [], [], []
    for s in test_samples:
        cfg = VortexRAGConfig(domain='financial', verbose=False)
        cfg.vrc.top_k = len(s['chunks'])
        cfg.rfg.top_m = 2
        cfg.cpg.theta_cpg = theta_cpg
        cfg.sdc.delta_sdc = delta_sdc
        rag = VortexRAG(config=cfg)
        rag.index(additional_texts=s['chunks'])
        r = rag.query(s['q'])
        f1s.append(token_f1(r.answer, s['gold']))
        esrs.append(r.esr)
        drs.append(r.delta_r)
    print(f'  {method_name:<13} {sum(f1s)/len(f1s):>8.3f} {sum(esrs)/len(esrs):>8.3f} {sum(drs)/len(drs):>10.3f}')


---
## Summary

VORTEXRAG's 7-layer architecture provides:
- **Semantic drift elimination** via SDC causal gate
- **Context poisoning prevention** via CPG ESR-based purification
- **No-weak-link ranking** via multiplicative Phi-score fusion
- **Causal ordering** via CCB slot injection
- **Hallucination detection** via FV Delta-R scoring

### Key Hyperparameters

| Parameter | Default | Effect |
|-----------|---------|--------|
| `delta_sdc` | 0.72 | SDC gate (higher = stricter drift rejection) |
| `tau` | domain-tuned | Drift temperature (lower = stricter) |
| `theta_cpg` | 3.5 | ESR threshold (higher = stricter poison guard) |
| `delta_fv` | 0.15 | Hallucination threshold (lower = stricter) |
| `n_spiral` | 2 | Spiral tightness (3 = most precise) |
| `lambda_decay` | 0.5 | Radial decay rate in VRC |

### Repository
**GitHub:** https://github.com/vignesh2027/VORTEXRAG
